# Paragraph extraction from page content

RAT loads the HTML content of a page using LangChain library's `AsyncHtmlLoader` class, which fails to load as some pages such as [gousa.in/destination/new-york-city](https://www.gousa.in/destination/new-york-city) require JavaScript.

RAT extracts HTML into text using LangChain library's `Html2TextTransformer` class, which returns all the content as text, without a focus on extracting the actual content. It becomes a major issue when a page has a lot of such content.

Example: https://www.pbs.org/wgbh/americanexperience/features/new-york-historic/
```
	Skip To Content

	Join the Conversation

	Facebook
	Twitter
	Instagram
	YouTube
	Funded by

	Support Provided by: Learn More Dismiss
```


Our new module extracts the content properly in such scenarios.

In [1]:
# display the md content in jupyter notebook
from IPython.display import Markdown, display

def display_markdown(content: str):
    display(Markdown(content))

from dotenv import load_dotenv
load_dotenv(override=True)

True

### Existing function from RAT

In [2]:
# from langchain_community.document_transformers import Html2TextTransformer
# from langchain_community.document_loaders import AsyncHtmlLoader

# def get_page_content(link: str):
# 	loader = AsyncHtmlLoader([link])
# 	docs = loader.load()
# 	html2text = Html2TextTransformer()
# 	docs_transformed = html2text.transform_documents(docs)
# 	if len(docs_transformed) > 0:
# 		return docs_transformed[0].page_content
# 	else:
# 		return None

# # Example usage:
# content = get_page_content('https://en.wikipedia.org/wiki/New_York_City')
# if content:
# 	display_markdown(content[:100])

## New approach

In [3]:
# !pip install playwright nest_asyncio
# !playwright install --with-deps chromium
# if it fails, try   !playwright install chromium

In [4]:
# Apply asyncio in a Jupyter notebook environment
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import asyncio
from langchain_community.document_loaders import AsyncHtmlLoader, AsyncChromiumLoader
# from langchain_community.document_transformers import Html2TextTransformer, MarkdownifyTransformer

# # text_transformer = MarkdownifyTransformer(autolinks=False)
# text_transformer = Html2TextTransformer(ignore_links=True, ignore_images=True)

# !pip install readability-lxml
from readability import Document as ReadabilityDocument
from bs4 import BeautifulSoup

class ParagraphExtractor:
	def __init__(self):
		self.h_tags = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']
		self.list_tags = ['li', 'ul', 'ol']
		self.tags = self.h_tags + self.list_tags + ['p', 'blockquote']

	def extract_main_paragraphs_from_html(self, doc: str) -> list[str]:
		reader = ReadabilityDocument(doc.page_content.strip())
		cleaned_html = reader.summary(html_partial=True)

		soup = BeautifulSoup(cleaned_html, 'html.parser')
		paragraphs = []
		for part in soup.find_all(self.tags):
			text = part.get_text().strip()
			if not text:
				continue

			if part.name in self.h_tags:
				n = int(part.name[1])  # Get the heading level
				text = f'{"#" * n} {text}'  # Markdown heading
			elif part.name in self.list_tags:
				if part.name == 'ol':
					text = f'1. {text}'
				else:
					text = f'- {text}'
			elif part.name == 'blockquote':
				text = f'> {text}'
			elif part.name == 'p':
				# Process <p> tag and handle bold/italic formatting
				text = ''
				for child in part.descendants:
					if child.name in ['b', 'strong']:
						text += f'**{child.get_text().strip()}**'
					elif child.name in ['i', 'em']:
						text += f'_{child.get_text().strip()}_'
					elif child.string:
						text += child.string.strip()
				text = text.strip()
			else:
				text = part.get_text().strip()

			paragraphs.append(text)
		content = '\n\n'.join(paragraphs)
		doc.page_content = content
		return doc

	def transform_documents(self, docs: list[str]) -> list[str]:
		return [self.extract_main_paragraphs_from_html(doc) for doc in docs]

text_transformer = ParagraphExtractor()

async def get_one_page_content_async(url: str, use_chromium: bool = False, only_html: bool = False):
	if not url:
		print('No URL provided')
		return None
	if use_chromium:
		print('Using Chromium loader for JavaScript rendering')
		try:
			loader = AsyncChromiumLoader([url])  # handles JavaScript rendering
		except Exception as e:
			print(f'Error initializing AsyncChromiumLoader: {e}')
			return None
	else:
		loader = AsyncHtmlLoader([url])

	docs = await loader.aload()
	if not docs:
		return None

	docs_transformed = text_transformer.transform_documents(docs)
	if not docs_transformed:
		return None
	page_content = docs_transformed[0].page_content.strip()
	if not page_content:
		return None

	# if parsing failed and JavaScript is required
	if page_content.startswith('Enable JavaScript'):
		if use_chromium:  # If JS failed even with Chromium
			raise ValueError('JavaScript rendering failed')
		# try using Chromium loader
		page_content = await get_one_page_content_async(url, use_chromium=True)
		if not page_content:
			return None
	if 'Verifying you are human' in page_content:
		print('Loading failed due to bot protection:', url)
		return None

	return page_content

# def get_page_content(url: str, use_chromium: bool = False, only_html: bool = False):
# 	return asyncio.run(get_page_content_async(url, use_chromium, only_html))

def get_page_contents(urls: list, use_chromium: bool = False, only_html: bool = False):
	if not urls or not isinstance(urls, list):
		print("Please provide a list of URLs.")
		return []

	results = []
	for url in urls:
		try:
			content = asyncio.run(get_one_page_content_async(url, use_chromium, only_html))
			results.append(content or None)  # pass None if content is empty
		except Exception as e:
			print(f"Error processing URL {url}: {e}")
			results.append(None)

	return results

In [ ]:
contents = get_page_contents(['https://www.pbs.org/wgbh/americanexperience/features/new-york-historic/'])
if contents:
	display_markdown(contents[0][:500])

Fetching pages: 100%|##########| 1/1 [00:01<00:00,  1.78s/it]


## Historic New York

The country's first capital, the site of America's first explosive urban expansion, and the place where modern media was invented, New York has been host to historic events that have had an unforgettable impact on the rest of America. Following are a few of the most notable.

**A Capital City**A Capital CityNew York was the first capital of the United States -- George Washington was sworn in as the first President on the balcony of New York City's old City Hall on April 30,